# End-to-End Machine Learning Prediction Project
## Flavors of Cacao – Chocolate Rating Prediction

**Student Register Number:** 241FA18155 
**Dataset:** `flavors_of_cacao.csv`  
**Problem Type:** Regression  
**Target Variable:** `Rating`

### Problem Statement
The objective is to predict the **chocolate bar rating** using information such as company, bean origin, review year, cocoa percentage, company location, bean type, and broad bean origin.

### Project Workflow
1. Understand the dataset and define the problem
2. Perform EDA
3. Handle missing values and duplicates
4. Inspect outliers
5. Perform feature engineering
6. Build multiple suitable regression models
7. Evaluate and compare the models
8. Tune the best suitable model
9. Evaluate the final model on unseen test data
10. Perform a sample prediction
11. Save the final model
12. Give a final conclusion

**Note:** This is a **regression** problem because the target `Rating` is a continuous numerical value.

## Import Required Libraries

We import libraries for data handling, visualization, preprocessing, model building, evaluation, hyperparameter tuning, and saving the final model.

**Why:** Keeping the imports together makes the notebook easier to understand and reproduce.

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

## 1. Load the Dataset

The CSV file is expected to be in the **same folder as this notebook**.

**Why:** Loading the original dataset is the first step before understanding or preprocessing the data.

**Expected output:** Dataset shape, column names, and the first few rows.

In [ ]:
data_path = os.path.join("data", "raw", "flavors_of_cacao.csv")

if not os.path.exists(data_path):
    raise FileNotFoundError(
        f"{data_path} was not found. Put the CSV file in the same folder as this notebook."
    )

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

## 2. Clean Column Names

The original dataset contains line breaks, extra spaces, and non-breaking spaces in some column names.

**Why:** Clean column names make the remaining Python code easier to read and reduce errors.

In [ ]:
df.columns = (
    df.columns
    .str.replace("\xa0", " ", regex=False)
    .str.replace("\n", " ", regex=False)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

print("Cleaned column names:")
print(df.columns.tolist())

## 3. Understand the Dataset

We inspect data types, dimensions, and descriptive statistics.

**Why:** This helps us identify numerical and categorical variables and decide which preprocessing methods are appropriate.

In [ ]:
print("Dataset information:")
df.info()

print("\nSummary statistics:")
display(df.describe(include="all").T)

## 4. Define the Target Variable

The target variable is **`Rating`**.

A rating is a numerical value, so the project is treated as a **regression problem**.

**Why:** Regression models predict continuous numerical values rather than class labels.

In [ ]:
target = "Rating"

print("Target variable:", target)
print("\nTarget summary:")
print(df[target].describe())

## 5. Exploratory Data Analysis – Rating Distribution

We visualize the distribution of chocolate ratings.

**Why:** The distribution helps us understand the typical rating range and whether the target is heavily concentrated in a particular region.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df[target].dropna(), bins=15)
plt.title("Distribution of Chocolate Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Reviews")
plt.tight_layout()
plt.show()

## 6. Exploratory Data Analysis – Cocoa Percentage

`Cocoa Percent` is stored as text such as `70%`.

**Why:** It must be converted into a numerical percentage before it can be used effectively by regression models.

In [ ]:
print("Sample Cocoa Percent values:")
display(df["Cocoa Percent"].head(10))

## 7. Feature Engineering – Cocoa Percentage

We create a numerical `Cocoa_Percent` feature by removing the `%` symbol and converting the value to a number.

**Why:** A numerical cocoa percentage can be directly used by machine learning models.

In [ ]:
df["Cocoa_Percent"] = (
    df["Cocoa Percent"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.strip()
    .replace({"": np.nan, "nan": np.nan})
)

df["Cocoa_Percent"] = pd.to_numeric(df["Cocoa_Percent"], errors="coerce")

display(df[["Cocoa Percent", "Cocoa_Percent"]].head())
print("Cocoa_Percent data type:", df["Cocoa_Percent"].dtype)

## 8. Feature Engineering – Review Year

The `Review Date` column represents the year in which the chocolate was reviewed.

We create `Review_Year` as a clean numerical feature.

**Why:** Review year may capture changes in chocolate quality, manufacturing, or reviewing patterns over time.

In [ ]:
df["Review_Year"] = pd.to_numeric(df["Review Date"], errors="coerce")

print("Review year range:")
print(df["Review_Year"].min(), "to", df["Review_Year"].max())

## 9. Check Missing Values

We count missing values in every column.

**Why:** Missing values can cause model training errors or reduce model quality. We will handle them using a preprocessing pipeline rather than manually filling values outside the pipeline.

In [ ]:
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_percent = (df.isnull().mean() * 100).sort_values(ascending=False)

missing_report = pd.DataFrame({
    "Missing Count": missing_values,
    "Missing Percentage": missing_percent.round(2)
})

display(missing_report)

## 10. Check Duplicate Records

Duplicate rows can give repeated observations extra influence during training.

**Why:** We check duplicates before model training so that repeated records do not unnecessarily affect the model.

In [ ]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicates removed.")
else:
    print("No duplicate rows found.")

## 11. Check Target Values

We inspect the minimum, maximum, mean, and unusual values of the target.

**Why:** This helps confirm that the target is numeric and lets us check whether there are values outside the expected rating range.

In [ ]:
print("Rating minimum:", df[target].min())
print("Rating maximum:", df[target].max())
print("Rating mean:", round(df[target].mean(), 3))

print("\nUnique rating values:")
print(sorted(df[target].dropna().unique())[:20])

## 12. Outlier Analysis

We use the IQR method to inspect possible outliers in important numerical columns.

**Why:** Outlier analysis is useful, but we should not automatically delete unusual observations. A real chocolate rating or cocoa percentage can be a valid observation.

In [ ]:
numeric_check_cols = ["REF", "Review_Year", "Cocoa_Percent", "Rating"]

outlier_report = []

for col in numeric_check_cols:
    series = df[col].dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((series < lower) | (series > upper)).sum()
    outlier_report.append([col, q1, q3, lower, upper, int(count)])

outlier_report = pd.DataFrame(
    outlier_report,
    columns=["Column", "Q1", "Q3", "Lower Bound", "Upper Bound", "Possible Outliers"]
)

display(outlier_report)

## 13. Explore Ratings by Cocoa Percentage

We visualize the relationship between cocoa percentage and rating.

**Why:** This gives an initial view of whether cocoa concentration appears to have a simple or complex relationship with the rating.

In [ ]:
plot_df = df[["Cocoa_Percent", "Rating"]].dropna()

plt.figure(figsize=(7, 4))
plt.scatter(plot_df["Cocoa_Percent"], plot_df["Rating"], alpha=0.6)
plt.title("Cocoa Percentage vs Chocolate Rating")
plt.xlabel("Cocoa Percentage")
plt.ylabel("Rating")
plt.tight_layout()
plt.show()

## 14. Explore Ratings by Review Year

We calculate the average rating for each review year.

**Why:** This helps us see whether average ratings vary across the years represented in the dataset.

In [ ]:
year_rating = df.groupby("Review_Year")["Rating"].mean().dropna()

plt.figure(figsize=(8, 4))
plt.plot(year_rating.index, year_rating.values, marker="o")
plt.title("Average Rating by Review Year")
plt.xlabel("Review Year")
plt.ylabel("Average Rating")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 15. Explore Ratings by Company Location

We show the average rating for the most common company locations.

**Why:** Company location may capture regional differences in chocolate production and company characteristics.

In [ ]:
location_rating = (
    df.groupby("Company Location")["Rating"]
    .agg(["mean", "count"])
    .sort_values("count", ascending=False)
    .head(15)
)

display(location_rating)

plt.figure(figsize=(9, 5))
plt.barh(location_rating.index[::-1], location_rating["mean"][::-1])
plt.title("Average Rating – Top Company Locations by Review Count")
plt.xlabel("Average Rating")
plt.tight_layout()
plt.show()

## 16. Prepare Features and Target

We remove the original target and helper text columns that have already been converted into useful numerical features.

`Cocoa_Percent` and `Review_Year` are retained as engineered features.

**Why:** The target must not be included among the input features because that would cause data leakage.

In [ ]:
drop_columns = [
    target,
    "Cocoa Percent",
    "Review Date"
]

X = df.drop(columns=drop_columns)
y = df[target]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeatures:")
print(X.columns.tolist())

## 17. Remove Rows with Missing Target

The model cannot learn from a row where the target rating itself is missing.

**Why:** We remove only rows with missing `Rating`. Missing values in input features are handled later by the preprocessing pipeline.

In [ ]:
valid_target = y.notna()

X = X.loc[valid_target].reset_index(drop=True)
y = y.loc[valid_target].reset_index(drop=True)

print("Rows available for modeling:", len(X))
print("Missing target values remaining:", y.isna().sum())

## 18. Identify Numerical and Categorical Features

Numerical columns will use median imputation and scaling.

Categorical columns will use most-frequent imputation and one-hot encoding.

**Why:** Machine learning models require numerical inputs, so categorical values must be encoded.

In [ ]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

## 19. Train-Test Split

We split the dataset into training and testing sets.

- **80%** is used for training.
- **20%** is kept as unseen test data.

**Why:** The test set gives an honest estimate of performance on new data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 20. Build the Preprocessing Pipeline

The preprocessing pipeline performs:

- Median imputation for numerical features
- Standard scaling for numerical features
- Most-frequent imputation for categorical features
- One-hot encoding for categorical features

**Why:** Putting preprocessing inside a pipeline prevents data leakage because transformations are learned only from the training data.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=2))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created.")

## 21. Model 1 – Linear Regression

Linear Regression is used as a simple baseline model.

**Why:** It provides an easy-to-understand reference point. If more complex models perform better, they capture relationships that a simple linear model cannot.

In [ ]:
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

linear_mae = mean_absolute_error(y_test, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))
linear_r2 = r2_score(y_test, linear_pred)

print("Linear Regression")
print("MAE :", round(linear_mae, 4))
print("RMSE:", round(linear_rmse, 4))
print("R²  :", round(linear_r2, 4))

## 22. Model 2 – Ridge Regression

Ridge Regression adds L2 regularization to Linear Regression.

**Why:** One-hot encoding creates many features. Ridge regularization can reduce the effect of overly large coefficients and improve generalization.

In [ ]:
ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=10.0))
])

ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)

ridge_mae = mean_absolute_error(y_test, ridge_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
ridge_r2 = r2_score(y_test, ridge_pred)

print("Ridge Regression")
print("MAE :", round(ridge_mae, 4))
print("RMSE:", round(ridge_rmse, 4))
print("R²  :", round(ridge_r2, 4))

## 23. Model 3 – Random Forest Regressor

Random Forest is an ensemble of decision trees.

**Why:** It can model nonlinear relationships and interactions between features without requiring us to manually define those relationships.

In [ ]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest Regressor")
print("MAE :", round(rf_mae, 4))
print("RMSE:", round(rf_rmse, 4))
print("R²  :", round(rf_r2, 4))

## 24. Model 4 – Gradient Boosting Regressor

Gradient Boosting builds trees sequentially, where later trees focus on correcting previous errors.

**Why:** It is a strong classical regression algorithm and can capture nonlinear patterns in tabular data.

In [ ]:
gb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=2,
        random_state=42
    ))
])

gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)

gb_mae = mean_absolute_error(y_test, gb_pred)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_pred))
gb_r2 = r2_score(y_test, gb_pred)

print("Gradient Boosting Regressor")
print("MAE :", round(gb_mae, 4))
print("RMSE:", round(gb_rmse, 4))
print("R²  :", round(gb_r2, 4))

## 25. Evaluate and Compare All Models

For regression, we use:

- **MAE:** Average absolute prediction error. Lower is better.
- **RMSE:** Penalizes larger errors more strongly. Lower is better.
- **R²:** Proportion of target variation explained by the model. Higher is better.

**Why:** Comparing several metrics gives a more complete picture than using accuracy, which is not appropriate for this regression problem.

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE": [
        linear_mae,
        ridge_mae,
        rf_mae,
        gb_mae
    ],
    "RMSE": [
        linear_rmse,
        ridge_rmse,
        rf_rmse,
        gb_rmse
    ],
    "R2": [
        linear_r2,
        ridge_r2,
        rf_r2,
        gb_r2
    ]
}).sort_values("R2", ascending=False).reset_index(drop=True)

display(results.round(4))

## 26. Hypothesis Testing – P-value and R²

To complement the machine-learning model comparison, an OLS statistical model is used to test whether the main numerical predictors have a statistically significant linear relationship with **Rating**.

### Hypotheses
- **H₀:** The predictor has no statistically significant linear relationship with Rating.
- **H₁:** The predictor has a statistically significant linear relationship with Rating.
- **Significance level (α): 0.05**.

### Decision rule
- **p-value < 0.05:** Reject H₀ → statistically significant predictor.
- **p-value ≥ 0.05:** Fail to reject H₀ → insufficient evidence of statistical significance.

**Important:** p-value and R² are not directly compared with each other. P-value is compared with **0.05**, while R² is used to measure explanatory power/model performance.

In [ ]:
# OLS hypothesis testing for the main numerical predictors
import statsmodels.api as sm

stat_features = ["REF", "Cocoa_Percent", "Review_Year"]
stat_data = pd.concat([X_train[stat_features], y_train.rename("Rating")], axis=1).dropna()

X_stat = sm.add_constant(stat_data[stat_features])
y_stat = stat_data["Rating"]

ols_model = sm.OLS(y_stat, X_stat).fit()

print("OLS Statistical Analysis")
print("------------------------")
print("OLS R²:", round(ols_model.rsquared, 4))
print("Significance level (α): 0.05")

pvalue_table = pd.DataFrame({
    "Coefficient": ols_model.params,
    "P-value": ols_model.pvalues,
    "Significant (p < 0.05)": ols_model.pvalues < 0.05
})

display(pvalue_table)


## 27. Visual Model Comparison

We compare R² values visually.

**Why:** A chart makes it easier to see which model explains the largest amount of variation in the test ratings.

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(results["Model"], results["R2"])
plt.title("Model Comparison – R² Score")
plt.xlabel("Model")
plt.ylabel("R²")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 28. Hyperparameter Tuning – Random Forest

`RandomizedSearchCV` tests different combinations of Random Forest hyperparameters using cross-validation.

**Why:** Tuning can improve generalization without repeatedly using the final test set for model selection.

In [ ]:
rf_pipeline_for_tuning = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ))
])

param_distributions = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 5, 10, 15, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": [0.5, 0.75, 1.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline_for_tuning,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="r2",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)

print("Best parameters:")
print(rf_search.best_params_)
print("\nBest cross-validation R²:", round(rf_search.best_score_, 4))

## 29. Evaluate the Tuned Random Forest on Unseen Test Data

The tuned model is now evaluated once on the test set that was kept aside earlier.

**Why:** This gives the final estimate of performance on unseen data.

In [ ]:
tuned_rf = rf_search.best_estimator_
tuned_rf_pred = tuned_rf.predict(X_test)

tuned_mae = mean_absolute_error(y_test, tuned_rf_pred)
tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_rf_pred))
tuned_r2 = r2_score(y_test, tuned_rf_pred)

print("Tuned Random Forest")
print("MAE :", round(tuned_mae, 4))
print("RMSE:", round(tuned_rmse, 4))
print("R²  :", round(tuned_r2, 4))

## 30. Final Model Comparison

We add the tuned Random Forest to the baseline model results.

**Why:** The tuned model should be compared against the untuned models before making the final selection.

In [ ]:
final_results = pd.concat([
    results,
    pd.DataFrame([{
        "Model": "Tuned Random Forest",
        "MAE": tuned_mae,
        "RMSE": tuned_rmse,
        "R2": tuned_r2
    }])
], ignore_index=True)

final_results = final_results.sort_values(
    by=["R2", "RMSE"],
    ascending=[False, True]
).reset_index(drop=True)

display(final_results.round(4))

## 31. Actual vs Predicted Ratings — Final Random Forest

We compare the actual test ratings with predictions from the **original Random Forest**, which is retained as the final model because it has the best unseen-test metrics.

**Why:** Points close to the diagonal line indicate more accurate predictions.


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, rf_pred, alpha=0.6)

min_value = min(y_test.min(), rf_pred.min())
max_value = max(y_test.max(), rf_pred.max())

plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")
plt.title("Actual vs Predicted Ratings – Final Random Forest")
plt.xlabel("Actual Rating")
plt.ylabel("Predicted Rating")
plt.tight_layout()
plt.show()


## 32. Residual Analysis — Final Random Forest

A residual is the difference between the actual rating and predicted rating. The residual analysis now uses the **final original Random Forest** predictions.

**Why:** Residuals centered around zero with no obvious systematic pattern support reasonable prediction behavior.


In [ ]:
residuals = y_test - rf_pred

plt.figure(figsize=(8, 4))
plt.scatter(rf_pred, residuals, alpha=0.6)
plt.axhline(0, linestyle="--")
plt.title("Residual Plot – Final Random Forest")
plt.xlabel("Predicted Rating")
plt.ylabel("Residual (Actual - Predicted)")
plt.tight_layout()
plt.show()


## 33. Feature Importance — Final Random Forest

Random Forest provides feature-importance values after preprocessing. The importances below are extracted from the **original Random Forest final model**.

**Why:** Feature importance helps identify which encoded inputs contributed most to the final model's predictions.


In [ ]:
fitted_preprocessor = rf_model.named_steps["preprocessor"]
fitted_rf = rf_model.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()
importances = fitted_rf.feature_importances_

feature_importance = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    })
    .sort_values("Importance", ascending=False)
    .head(20)
)

display(feature_importance)

plt.figure(figsize=(9, 7))
plt.barh(
    feature_importance["Feature"][::-1],
    feature_importance["Importance"][::-1]
)
plt.title("Top 20 Feature Importances – Final Random Forest")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 34. Cross-Validation / Tuning Check

Hyperparameter tuning was performed for Random Forest using cross-validation, but the tuned model is **not automatically chosen**. Its unseen-test metrics must still be compared with the original model.

The tuned model achieved a cross-validation R² reported below, but its test R² (**0.2255**) is slightly below the original Random Forest test R² (**0.2314**). Therefore, the original Random Forest remains the final model.


In [ ]:
print("Best tuned-model cross-validation R²:", round(rf_search.best_score_, 4))
print("Original Random Forest test R²:", round(rf_r2, 4))
print("Tuned Random Forest test R²:", round(tuned_r2, 4))

if rf_r2 >= tuned_r2:
    print("Decision: Keep the original Random Forest as the final model.")
else:
    print("Decision: Use the tuned Random Forest as the final model.")


## 35. Sample Prediction on New / Unseen Data

A new chocolate example is passed to the **final original Random Forest model**.

**Why:** This demonstrates how the selected model can be used on unseen input after model comparison is complete.


In [ ]:
sample_input = pd.DataFrame([{
    "Company (Maker-if known)": "A. Morin",
    "Specific Bean Origin or Bar Name": "Sample Origin",
    "REF": 2000,
    "Company Location": "France",
    "Bean Type": np.nan,
    "Broad Bean Origin": "Peru",
    "Cocoa_Percent": 70.0,
    "Review_Year": 2020
}])

sample_prediction = rf_model.predict(sample_input)[0]

print("Predicted chocolate rating:", round(float(sample_prediction), 3))


## 36. Save the Final Model

We save the complete **original Random Forest pipeline**, including preprocessing and the fitted Random Forest model.

**Why:** This is the model that achieved the best reported unseen-test R², MAE and RMSE.


In [ ]:
model_filename = "241FA18155_flavors_of_cacao_model.joblib"

joblib.dump(rf_model, model_filename)

print("Final model saved as:", model_filename)
print("Saved model: Original Random Forest Regressor")


## 37. Load the Saved Model and Test It

We load the saved model from disk and use it to predict the same unseen sample.

**Why:** This confirms that the saved model can be reused after the notebook session ends.

In [ ]:
loaded_model = joblib.load(model_filename)
loaded_prediction = loaded_model.predict(sample_input)[0]

print("Prediction from loaded model:", round(float(loaded_prediction), 3))

## 38. Final Model Summary

The final model is the **Random Forest Regressor (original / untuned)**.

It is selected after considering all relevant predictive metrics:

- **R²:** Random Forest = **0.2314**, the highest among the reported models.
- **MAE:** Random Forest = **0.3259**, the lowest among the reported models.
- **RMSE:** Random Forest = **0.4215**, the lowest among the reported models.
- **Tuned Random Forest:** R² = **0.2255**, MAE = **0.3298**, RMSE = **0.4231**, so tuning did not improve unseen-test performance.

The OLS hypothesis test is considered separately for statistical significance. At **α = 0.05**, a predictor with **p-value < 0.05** is statistically significant. These p-values are not predictive scores for Random Forest, Ridge, or Gradient Boosting and therefore are not directly combined with their R² values.

### Final selection scenario

Because the original Random Forest has the **highest R² and the lowest MAE and RMSE**, and the tuned version does not improve these unseen-test metrics, the **original Random Forest is retained as the final predictive model**.


In [ ]:
print("FINAL MODEL: Random Forest Regressor (Original / Untuned)")
print("-" * 58)
print("MAE :", round(rf_mae, 4))
print("RMSE:", round(rf_rmse, 4))
print("R²  :", round(rf_r2, 4))
print("Tuned RF R² for comparison:", round(tuned_r2, 4))
print("Best tuned CV R²:", round(rf_search.best_score_, 4))


# 39. Final Conclusion

This project developed an end-to-end **regression machine learning system for predicting chocolate ratings** from the Flavors of Cacao dataset.

### Final comparison conclusion

Four baseline regression models were compared: Linear Regression, Ridge Regression, Random Forest, and Gradient Boosting. Random Forest achieved the strongest baseline test performance with **R² = 0.2314, MAE = 0.3259, and RMSE = 0.4215**.

Random Forest was then hyperparameter-tuned. The tuned model achieved **R² = 0.2255, MAE = 0.3298, and RMSE = 0.4231** on the unseen test set. Because those values are slightly worse than the original Random Forest, tuning did not improve final test performance. Therefore, the **original Random Forest Regressor is the final model**.

The OLS statistical analysis is used separately to interpret predictor significance using **p-value < 0.05** at α = 0.05. P-values support statistical interpretation, while **R², MAE and RMSE are used for final predictive-model selection**.

The complete final Random Forest pipeline is saved so the same preprocessing and prediction process can be reused for future observations.
